# HCFIM EEG Paper - Reproducible Notebook

This notebook supports the manuscript:

**Information-Theoretic Geometry and Lateralized HCFIM Features for
Data-Limited EEG Screening of Alzheimer's Disease and Frontotemporal Dementia**

It is intentionally independent from the operational Nouha `eeg-classifier`
deployment. The analysis uses the public OpenNeuro/NEMAR EEG dataset and its
preprocessed derivatives.

This notebook is organized in two parts:

1. **Methods figures:** synthetic regimes are projected into the probability
   simplex, HC plane, HFIM plane and HCFIM space. This produces the draft
   manuscript `[Figure 1]`.
2. **OpenNeuro experiments:** preprocessed EEG signals are indexed, segmented,
   transformed into probability distributions, projected into HCFIM features,
   compared against PSD baselines and evaluated with subject-level
   leakage-aware classification.

The default execution mode generates the synthetic methods figure and a small
EEG pilot only. Full feature extraction and classification are controlled by
explicit run flags below.


## 1. Setup

The notebook expects this project structure:

```text
HCFIM-eeg-research/
  src/
  notebooks/
  figures/
  results/
  manuscript/
  references/
```

Required Python packages include `numpy`, `pandas`, `scipy`, `matplotlib`,
`seaborn`, `scikit-learn`, `mne` and `pyarrow` or `fastparquet` if Parquet
output is used.


In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import seaborn as sns

from scipy import signal, stats
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from matplotlib.ticker import FormatStrFormatter

warnings.filterwarnings("ignore", category=RuntimeWarning)

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / "src"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"

for path in [SRC_DIR, FIGURES_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import information_theory_measures as itm
import itm_algorithmic_complexity as itm_ac
import itm_geometry as geom

RANDOM_STATE = 17536
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_theme(style="whitegrid", font="Arial")

PROJECT_ROOT

## 2. Analysis Parameters

`RUN_SYNTHETIC_METHOD_FIGURE` is enabled by default because it is lightweight
and generates the methods figure. Full EEG feature extraction is intentionally
disabled by default.


In [ ]:
def resolve_dataset_root(project_root: Path) -> Path:
    """Locate ds004504 derivatives or use HCFIM_DATASET_ROOT."""
    candidates = []
    env_root = os.environ.get("HCFIM_DATASET_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([
        project_root / "data" / "ds004504",
        project_root / "data" / "data-08-00095",
        project_root / "data",
    ])
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "participants.tsv").is_file() and (candidate / "derivatives").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate ds004504. Set HCFIM_DATASET_ROOT to the dataset directory "
        "containing participants.tsv and derivatives/."
    )

DATASET_ROOT = resolve_dataset_root(PROJECT_ROOT)
DERIVATIVES_ROOT = DATASET_ROOT / "derivatives"
PARTICIPANTS_TSV = DATASET_ROOT / "participants.tsv"

RUN_SYNTHETIC_METHOD_FIGURE = True
RUN_EEG_PILOT = False
RUN_FULL_EEG_FEATURE_EXTRACTION = False
RUN_FULL_CLASSIFICATION = True
RUN_DERIVED_APPENDIX_FEATURES = True

PILOT_SUBJECTS_PER_GROUP = 2
MAX_EPOCHS_PER_SUBJECT_PILOT = 4
MAX_EPOCHS_PER_SUBJECT_FULL = None

EPOCH_LENGTH_S = 10.0
EPOCH_OVERLAP = 0.50

CORE_BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "broadband": (0.5, 40.0),
}

REGIONS = {
    "frontal": ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8"],
    "temporal": ["T7", "T8", "P7", "P8"],
    "central": ["C3", "Cz", "C4"],
    "parietal": ["P3", "Pz", "P4"],
    "occipital": ["O1", "O2"],
}

REGIONS_LATERALIZED = {
    "frontal_left": ["Fp1", "F7", "F3"],
    "frontal_right": ["Fp2", "F8", "F4"],
    "temporal_left": ["T7", "P7"],
    "temporal_right": ["T8", "P8"],
    "parietal_left": ["P3"],
    "parietal_right": ["P4"],
    "occipital_left": ["O1"],
    "occipital_right": ["O2"],
    "central_midline_extended": ["C3", "Cz", "C4", "Fz", "Pz"],
}

CHANNEL_RENAME = {
    "T3": "T7",
    "T4": "T8",
    "T5": "P7",
    "T6": "P8",
}

GROUP_MAP = {
    "A": "alzheimer",
    "F": "frontotemporal_dementia",
    "C": "control",
}

SIGNAL_ORDER = [
    "stochastic",
    "oscillation",
    "chaotic",
    "periodic",
    "white_noise",
    "blue_noise",
    "violet_noise",
    "brownian_noise",
    "pink_noise",
]

CLASSIFICATION_COMPARISONS = {
    "control_vs_alzheimer": ["control", "alzheimer"],
    "control_vs_frontotemporal_dementia": ["control", "frontotemporal_dementia"],
}

PARAMS = {
    "dataset_root": str(DATASET_ROOT),
    "derivatives_root": str(DERIVATIVES_ROOT),
    "epoch_length_s": EPOCH_LENGTH_S,
    "epoch_overlap": EPOCH_OVERLAP,
    "bands": CORE_BANDS,
    "regions": REGIONS,
    "lateralized_regions": REGIONS_LATERALIZED,
    "random_state": RANDOM_STATE,
}

with open(RESULTS_DIR / "analysis_parameters.json", "w", encoding="utf-8") as f:
    json.dump(PARAMS, f, indent=2)

PARAMS

## 3. Synthetic Geometry For Methods Figure

This section generates the manuscript methods figure:

- **Figure 1a:** probability simplex induced by ordinal PDFs.
- **Figure 1b:** HC plane with Cmin/Cmax curves.
- **Figure 1c:** HFIM plane.
- **Figure 1d:** HCFIM space.

The first version uses Bandt-Pompe ordinal probabilities with `m = 3`,
`tau = 1`, permutation/Shannon entropy and Jensen-Shannon complexity. This is
deliberately low-dimensional so the simplex can be visualized.


In [ ]:
def ordinal_pdf_rows_for_synthetic(spec: geom.EntropySpec, n=600, replicates=36) -> pd.DataFrame:
    examples = geom.synthetic_signal_examples(
        n=n,
        replicates=replicates,
        random_state=RANDOM_STATE,
    )
    rows = []
    for row in examples.itertuples(index=False):
        probabilities = geom.ordinal_probabilities(row.values, m=int(spec.m), tau=int(spec.tau))
        hcfim = geom.hcfim_point(probabilities, spec, distance_code=geom.DISTANCES["jensen_shannon"]["code"])
        rows.append({
            "signal": row.signal,
            "replicate": row.replicate,
            "probabilities": probabilities,
            **hcfim,
        })
    output = pd.DataFrame(rows)
    output["signal"] = pd.Categorical(output["signal"], categories=SIGNAL_ORDER, ordered=True)
    return output


def build_simplex_projection(pdf_points: pd.DataFrame, spec: geom.EntropySpec, n_background=5000):
    rng = np.random.default_rng(RANDOM_STATE)
    n_states = int(spec.n_states)
    background = rng.dirichlet(np.ones(n_states), size=n_background)
    pure_states = np.eye(n_states)
    uniform = np.full((1, n_states), 1.0 / n_states)
    signal_pdfs = np.vstack(pdf_points["probabilities"].to_numpy())

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    all_for_fit = np.vstack([background, pure_states, uniform, signal_pdfs])
    pca.fit(all_for_fit)

    background_xy = pca.transform(background)
    pure_xy = pca.transform(pure_states)
    uniform_xy = pca.transform(uniform)
    signal_xy = pca.transform(signal_pdfs)

    projected = pdf_points.drop(columns=["probabilities"]).copy()
    projected["PC1"] = signal_xy[:, 0]
    projected["PC2"] = signal_xy[:, 1]
    return {
        "pca": pca,
        "background_xy": background_xy,
        "pure_xy": pure_xy,
        "uniform_xy": uniform_xy,
        "points": projected,
    }


def plot_methods_figure(spec: geom.EntropySpec, output_path: Path):
    pdf_points = ordinal_pdf_rows_for_synthetic(spec)
    simplex = build_simplex_projection(pdf_points, spec)
    hc_points = geom.synthetic_hc_points(
        spec,
        distance_names=["jensen_shannon"],
        n=600,
        replicates=36,
        random_state=RANDOM_STATE,
    )
    curves = geom.boundary_curves(spec, "jensen_shannon", min_size=300, max_m=80)

    fig = plt.figure(figsize=(16, 13.2), constrained_layout=False)
    plt.rcParams.update({'font.size': 14})
    plt.rcParams.update({
        'axes.titlesize': 16,     # Title size
        'axes.labelsize': 15,     # X and Y label size
        'xtick.labelsize': 15,    # X-tick size
        'ytick.labelsize': 15,    # Y-tick size
        'legend.fontsize': 13,    # Legend size
        'figure.titlesize': 18    # Super title size
    })
    gs = fig.add_gridspec(2, 2, wspace=0.15, hspace=0.15)

    # a) Induced simplex projection
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.scatter(simplex["background_xy"][:, 0], simplex["background_xy"][:, 1],
                s=5, c="#aaaaaa", alpha=0.20, linewidths=0, label="simplex samples")
    ax1.scatter(simplex["pure_xy"][:, 0], simplex["pure_xy"][:, 1],
                marker="^", s=95, c="#222222", label="pure states")
    ax1.scatter(simplex["uniform_xy"][:, 0], simplex["uniform_xy"][:, 1],
                s=95, facecolors="white", edgecolors="#222222", linewidths=1.8, label="uniform")

    simplex_points = simplex["points"]
    for signal_name in SIGNAL_ORDER:
        group = simplex_points[simplex_points["signal"] == signal_name]
        if not group.empty:
            ax1.scatter(group["PC1"], group["PC2"], s=35,
                        c=geom.SIGNAL_COLORS.get(signal_name, "#666666"),
                        alpha=0.65, label=signal_name)
    ax1.axhline(0, color="#bbbbbb", linewidth=0.9)
    ax1.axvline(0, color="#bbbbbb", linewidth=0.9)
    ax1.set_xlabel(f"PC1 of ordinal PDF ({simplex['pca'].explained_variance_ratio_[0] * 100:.1f}% var.)")
    ax1.set_ylabel(f"PC2 of ordinal PDF ({simplex['pca'].explained_variance_ratio_[1] * 100:.1f}% var.)")
    ax1.text(0.01, 0.98, "a", transform=ax1.transAxes, fontsize=18, fontweight="normal", va="top")
    ax1.tick_params(axis="both", labelsize=13)
    ax1.grid(False)

    # b) HC plane
    ax2 = fig.add_subplot(gs[0, 1])
    for curve_name, linestyle in [("Cmin", "--"), ("Cmax", "-")]:
        curve = curves[curves["curve"] == curve_name]
        ax2.plot(curve["H"], curve["C"], color="#777777", linestyle=linestyle, linewidth=1.2, label=curve_name)
    for signal_name in SIGNAL_ORDER:
        group = hc_points[hc_points["signal"] == signal_name]
        if not group.empty:
            ax2.scatter(group["H"], group["C"], s=35,
                        c=geom.SIGNAL_COLORS.get(signal_name, "#666666"),
                        alpha=0.65, label=signal_name)
    ax2.set_xlim(0, 1.0)
    ax2.set_xlabel(f"Normalized permutation entropy (H), m={spec.m}")
    ax2.set_ylabel("Statistical complexity (C_JS)")
    ax2.text(0.01, 0.98, "b", transform=ax2.transAxes, fontsize=18, fontweight="normal", va="top")
    ax2.tick_params(axis="both", labelsize=13)
    ax2.grid(False)
    
    # c) HFIM plane
    ax3 = fig.add_subplot(gs[1, 0])
    for signal_name in SIGNAL_ORDER:
        group = pdf_points[pdf_points["signal"] == signal_name]
        if not group.empty:
            ax3.scatter(group["H"], group["FIM"], s=35,
                        c=geom.SIGNAL_COLORS.get(signal_name, "#666666"),
                        alpha=0.65, label=signal_name)
    ax3.set_xlabel(f"Normalized permutation entropy (H), m={spec.m}")
    ax3.set_ylabel("Fisher Information Measure (FIM)")
    ax3.text(0.01, 0.98, "c", transform=ax3.transAxes, fontsize=18, fontweight="normal", va="top")
    ax3.tick_params(axis="both", labelsize=13)
    ax3.grid(False)
    
    # d) HCFIM space
    ax4 = fig.add_subplot(gs[1, 1], projection="3d")
    for signal_name in SIGNAL_ORDER:
        group = pdf_points[pdf_points["signal"] == signal_name]
        if not group.empty:
            ax4.scatter(group["H"], group["C"], group["FIM"],
                        s=30, c=geom.SIGNAL_COLORS.get(signal_name, "#666666"),
                        alpha=0.65, label=signal_name)            
    ax4.set_xlabel("H")
    ax4.set_ylabel("C_JS")
    ax4.set_zlabel("FIM")

    ax4.xaxis.pane.set_facecolor((1, 1, 1, 1))
    ax4.yaxis.pane.set_facecolor((1, 1, 1, 1))
    ax4.zaxis.pane.set_facecolor((1, 1, 1, 1))
    
    try:
        ax4.set_box_aspect((1.25, 1.0, 0.92), zoom=1.28)
    except TypeError:
        ax4.set_box_aspect((1.25, 1.0, 0.92))
    ax4.view_init(elev=23, azim=-54)
    ax4.tick_params(axis="both", labelsize=13)
    ax4.text2D(0.01, 0.98, "d", transform=ax4.transAxes, fontsize=18, fontweight="normal", va="top")

    legend_handles = [
        Line2D([0], [0], color="#777777", linestyle="--", linewidth=1.2, label="Cmin"),
        Line2D([0], [0], color="#777777", linestyle="-", linewidth=1.2, label="Cmax"),
        Line2D([0], [0], marker=".", linestyle="", color="#d9d9d9", alpha=0.45, markersize=9, label="simplex samples"),
        Line2D([0], [0], marker="^", linestyle="", color="#222222", markersize=8, label="pure stateLs"),
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor="white", markeredgecolor="#222222", markeredgewidth=1.4, color="#222222", markersize=8, label="uniform"),
    ]
    legend_handles.extend(
        Line2D([0], [0], marker="o", linestyle="", color=geom.SIGNAL_COLORS.get(signal_name, "#666666"),
               markersize=7, alpha=0.75, label=signal_name)
        for signal_name in SIGNAL_ORDER
    )
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.012),
        ncol=7,
        frameon=True,
        fontsize=13,
        handletextpad=0.4,
        columnspacing=1.0,
    )
    
    fig.suptitle("")
    fig.subplots_adjust(left=0.07, right=0.985, top=0.985, bottom=0.120)
    output_path.parent.mkdir(parents=True, exist_ok=True)    
    fig.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0.45)
    plt.show()
    return output_path, pdf_points, hc_points, curves


if RUN_SYNTHETIC_METHOD_FIGURE:
    permen_spec = next(spec for spec in geom.default_entropy_specs(m=3, tau=1)
                       if spec.entropy_id == "permutation_entropy")
    figure_1_path, synthetic_hcfim_points, synthetic_hc_points, hc_curves = plot_methods_figure(
        permen_spec,
        FIGURES_DIR / "figure_1_methods_synthetic_geometry.png",
    )
    synthetic_hcfim_points.drop(columns=["probabilities"]).to_csv(
        RESULTS_DIR / "synthetic_hcfim_points_figure_1.csv",
        index=False,
    )
    synthetic_hc_points.to_csv(RESULTS_DIR / "synthetic_hc_points_figure_1.csv", index=False)
    hc_curves.to_csv(RESULTS_DIR / "hc_boundary_curves_figure_1.csv", index=False)
    print(f"Saved {figure_1_path}")


## 4. OpenNeuro Dataset Index

This section creates a subject registry from `participants.tsv` and links each
participant to the preprocessed EEGLAB `.set` derivative.


In [ ]:
def load_participants(participants_tsv: Path) -> pd.DataFrame:
    participants = pd.read_csv(participants_tsv, sep="\t")
    participants = participants.rename(columns={
        "Gender": "gender",
        "Age": "age",
        "Group": "group_code",
    })
    participants["diagnosis"] = participants["group_code"].map(GROUP_MAP)
    participants["set_path"] = participants["participant_id"].apply(
        lambda sid: DERIVATIVES_ROOT / sid / "eeg" / f"{sid}_task-eyesclosed_eeg.set"
    )
    participants["set_exists"] = participants["set_path"].apply(lambda p: Path(p).exists())
    return participants


participants = load_participants(PARTICIPANTS_TSV)
participants_summary = (
    participants
    .groupby("diagnosis", dropna=False)
    .agg(n=("participant_id", "count"),
         mean_age=("age", "mean"),
         sd_age=("age", "std"),
         mean_mmse=("MMSE", "mean"),
         sd_mmse=("MMSE", "std"),
         available_sets=("set_exists", "sum"))
    .reset_index()
)

participants.to_csv(RESULTS_DIR / "participants_index.csv", index=False)
participants_summary.to_csv(RESULTS_DIR / "participants_summary.csv", index=False)
participants_summary

## 5. EEG Loading And Feature Extraction Functions

The following functions are intentionally explicit. They define exactly how
preprocessed EEG derivatives are loaded, segmented, transformed into
probability distributions and summarized as subject-level features.


In [ ]:
def require_mne():
    try:
        import mne
    except Exception as exc:
        raise ImportError(
            "MNE is required to read EEGLAB .set files. Install with `pip install mne` "
            "in the notebook environment."
        ) from exc
    return mne


def canonical_channel_name(name: str) -> str:
    base = str(name).strip()
    return CHANNEL_RENAME.get(base, base)


def load_eeglab_subject(set_path: Path):
    mne = require_mne()
    raw = mne.io.read_raw_eeglab(str(set_path), preload=True, verbose="ERROR")
    rename_map = {ch: canonical_channel_name(ch) for ch in raw.ch_names if canonical_channel_name(ch) != ch}
    if rename_map:
        raw.rename_channels(rename_map)
    raw.pick_types(eeg=True)
    data = raw.get_data()
    channels = list(raw.ch_names)
    sfreq = float(raw.info["sfreq"])
    return data, sfreq, channels


def epoch_slices(n_samples: int, sfreq: float, epoch_length_s: float, overlap: float):
    window = int(round(epoch_length_s * sfreq))
    step = int(round(window * (1.0 - overlap)))
    if window <= 0 or step <= 0:
        raise ValueError("Invalid epoching parameters.")
    for epoch_id, start in enumerate(range(0, n_samples - window + 1, step)):
        yield epoch_id, start, start + window


def bandpass_1d(x: np.ndarray, sfreq: float, band: tuple[float, float]) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    x = signal.detrend(x, type="constant")
    low, high = band
    nyq = sfreq / 2.0
    low = max(float(low), 0.01)
    high = min(float(high), nyq - 0.5)
    if high <= low:
        return x
    sos = signal.butter(4, [low / nyq, high / nyq], btype="bandpass", output="sos")
    return signal.sosfiltfilt(sos, x)


def region_signal(window: np.ndarray, channels: list[str], region_channels: list[str]) -> np.ndarray | None:
    channel_index = {canonical_channel_name(ch): idx for idx, ch in enumerate(channels)}
    indices = [channel_index[ch] for ch in region_channels if ch in channel_index]
    if not indices:
        return None
    return np.nanmean(window[indices, :], axis=0)


def relative_bandpower_features(x: np.ndarray, sfreq: float, region_name: str) -> list[dict]:
    nperseg = min(int(round(sfreq * 2.0)), x.size)
    if nperseg < 8:
        return []
    freqs, psd = signal.welch(x, fs=sfreq, nperseg=nperseg, detrend="constant")
    total_mask = (freqs >= 0.5) & (freqs < 40.0)
    total_power = np.trapz(psd[total_mask], freqs[total_mask]) if np.any(total_mask) else np.nan
    if not np.isfinite(total_power) or total_power <= 0:
        return []

    rows = []
    band_values = {}
    for band_name, band in CORE_BANDS.items():
        if band_name == "broadband":
            continue
        mask = (freqs >= band[0]) & (freqs < band[1])
        power = np.trapz(psd[mask], freqs[mask]) if np.any(mask) else np.nan
        band_values[band_name] = float(power / total_power) if np.isfinite(power) else np.nan
        rows.append({
            "feature_family": "psd",
            "feature_name": f"psd__{region_name}__relative_bandpower_{band_name}",
            "value": band_values[band_name],
        })

    alpha = band_values.get("alpha", np.nan)
    beta = band_values.get("beta", np.nan)
    theta = band_values.get("theta", np.nan)
    rows.extend([
        {
            "feature_family": "psd",
            "feature_name": f"psd__{region_name}__theta_alpha_ratio",
            "value": float(theta / alpha) if alpha and alpha > 0 else np.nan,
        },
        {
            "feature_family": "psd",
            "feature_name": f"psd__{region_name}__alpha_beta_ratio",
            "value": float(alpha / beta) if beta and beta > 0 else np.nan,
        },
    ])

    psd_total = psd[total_mask]
    probs = psd_total / np.sum(psd_total)
    probs = np.clip(probs, np.finfo(float).tiny, 1.0)
    spectral_entropy = -np.sum(probs * np.log2(probs)) / np.log2(probs.size)
    rows.append({
        "feature_family": "psd",
        "feature_name": f"psd__{region_name}__spectral_entropy",
        "value": float(spectral_entropy),
    })
    return rows


TRANSFORMS_MAIN = [
    {"transform_id": "ordinal_shannon_m3_t1", "kind": "ordinal", "entropy_kind": "shannon", "m": 3, "tau": 1},
    {"transform_id": "ordinal_shannon_m4_t1", "kind": "ordinal", "entropy_kind": "shannon", "m": 4, "tau": 1},
    {"transform_id": "ordinal_shannon_m5_t1", "kind": "ordinal", "entropy_kind": "shannon", "m": 5, "tau": 1},
    {"transform_id": "ordinal_renyi_a08_m3_t1", "kind": "ordinal", "entropy_kind": "renyi", "alpha": 0.8, "m": 3, "tau": 1},
    {"transform_id": "ordinal_renyi_a12_m3_t1", "kind": "ordinal", "entropy_kind": "renyi", "alpha": 1.2, "m": 3, "tau": 1},
    {"transform_id": "ordinal_tsallis_q08_m3_t1", "kind": "ordinal", "entropy_kind": "tsallis", "q": 0.8, "m": 3, "tau": 1},
    {"transform_id": "ordinal_tsallis_q12_m3_t1", "kind": "ordinal", "entropy_kind": "tsallis", "q": 1.2, "m": 3, "tau": 1},
]

TRANSFORMS_DERIVED_APPENDIX = [
    {"transform_id": "apen_distance_m2_t1_b16", "kind": "derived", "derived_id": "apen_distance_pdf", "m": 2, "tau": 1, "bins": 16},
    {"transform_id": "sampen_distance_m2_t1_b16", "kind": "derived", "derived_id": "sampen_distance_pdf", "m": 2, "tau": 1, "bins": 16},
    {"transform_id": "fuzzy_similarity_m2_t1_b16", "kind": "derived", "derived_id": "fuzzy_similarity_pdf", "m": 2, "tau": 1, "bins": 16},
    {"transform_id": "k2_correlation_m2_t1_b16", "kind": "derived", "derived_id": "k2_correlation_pdf", "m": 2, "tau": 1, "bins": 16},
    {"transform_id": "lz_ordinal_phrase_m3_t1_b16", "kind": "derived", "derived_id": "lz_ordinal_phrase_pdf", "m": 3, "tau": 1, "bins": 16},
]


def entropy_spec_for_transform(transform: dict, n_states: int) -> geom.EntropySpec:
    return geom.EntropySpec(
        entropy_id=transform["transform_id"],
        label=transform["transform_id"],
        family=transform["kind"],
        probability_domain=transform.get("derived_id", "Bandt-Pompe ordinal patterns"),
        n_states=int(n_states),
        entropy_kind=transform.get("entropy_kind", "shannon"),
        m=transform.get("m"),
        tau=transform.get("tau"),
        q=transform.get("q"),
        alpha=transform.get("alpha"),
    )


def probability_for_transform(x: np.ndarray, transform: dict):
    if transform["kind"] == "ordinal":
        m = int(transform.get("m", 3))
        tau = int(transform.get("tau", 1))
        probabilities = geom.ordinal_probabilities(x, m=m, tau=tau)
        return probabilities, entropy_spec_for_transform(transform, n_states=math.factorial(m))

    if transform["kind"] == "derived":
        probabilities = itm_ac.derived_probability_vector(
            x,
            derived_id=transform["derived_id"],
            m=int(transform.get("m", 2)),
            tau=int(transform.get("tau", 1)),
            bins=int(transform.get("bins", 16)),
        )
        return probabilities, entropy_spec_for_transform(transform, n_states=len(probabilities))

    raise ValueError(f"Unknown transform kind: {transform['kind']}")


def hcfim_feature_rows(x: np.ndarray, region_name: str, band_name: str, transforms: list[dict]) -> list[dict]:
    rows = []
    for transform in transforms:
        try:
            probabilities, spec = probability_for_transform(x, transform)
            for distance_name, distance_meta in geom.DISTANCES.items():
                point = geom.hcfim_point(probabilities, spec, distance_code=distance_meta["code"])
                for coord_name in ["H", "C", "FIM"]:
                    rows.append({
                        "feature_family": "hcfim",
                        "feature_name": (
                            f"hcfim__{transform['transform_id']}__{distance_name}"
                            f"__{region_name}__{band_name}__{coord_name}"
                        ),
                        "value": float(point[coord_name]),
                    })
        except Exception as exc:
            rows.append({
                "feature_family": "hcfim_error",
                "feature_name": f"hcfim_error__{transform['transform_id']}__{region_name}__{band_name}",
                "value": np.nan,
                "error": f"{type(exc).__name__}: {exc}",
            })
    return rows

## 6. Extract Epoch-Level Features

For the first scientific notebook, extraction is deliberately written as a
transparent loop. If the full analysis becomes slow, the same logic can later
be moved into a parallelized script while preserving the saved schemas.

In [ ]:
def select_balanced_pilot(participants: pd.DataFrame, n_per_group: int) -> pd.DataFrame:
    return (
        participants[participants["set_exists"]]
        .sort_values(["diagnosis", "participant_id"])
        .groupby("diagnosis", group_keys=False)
        .head(n_per_group)
        .reset_index(drop=True)
    )


def extract_epoch_features_for_subject(
    subject_row,
    regions: dict[str, list[str]],
    transforms: list[dict],
    max_epochs: int | None = None,
) -> pd.DataFrame:
    data, sfreq, channels = load_eeglab_subject(Path(subject_row.set_path))
    rows = []
    for epoch_id, start, end in epoch_slices(data.shape[1], sfreq, EPOCH_LENGTH_S, EPOCH_OVERLAP):
        if max_epochs is not None and epoch_id >= int(max_epochs):
            break
        window = data[:, start:end]
        base = {
            "participant_id": subject_row.participant_id,
            "diagnosis": subject_row.diagnosis,
            "group_code": subject_row.group_code,
            "epoch_id": epoch_id,
            "start_sample": start,
            "end_sample": end,
            "sfreq": sfreq,
        }
        for region_name, region_channels in regions.items():
            x_region = region_signal(window, channels, region_channels)
            if x_region is None:
                continue

            for psd_row in relative_bandpower_features(x_region, sfreq, region_name):
                rows.append({**base, "region": region_name, "band": "broadband", **psd_row})

            for band_name, band in CORE_BANDS.items():
                x_band = bandpass_1d(x_region, sfreq, band)
                for feature_row in hcfim_feature_rows(x_band, region_name, band_name, transforms):
                    rows.append({**base, "region": region_name, "band": band_name, **feature_row})
    return pd.DataFrame(rows)


def extract_feature_table(
    selected_subjects: pd.DataFrame,
    regions: dict[str, list[str]],
    transforms: list[dict],
    max_epochs_per_subject: int | None,
    output_name: str,
) -> pd.DataFrame:
    frames = []
    for idx, subject_row in enumerate(selected_subjects.itertuples(index=False), start=1):
        print(f"[{idx}/{len(selected_subjects)}] Extracting {subject_row.participant_id} ({subject_row.diagnosis})")
        frames.append(
            extract_epoch_features_for_subject(
                subject_row,
                regions=regions,
                transforms=transforms,
                max_epochs=max_epochs_per_subject,
            )
        )
    features = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    csv_path = RESULTS_DIR / f"{output_name}.csv"
    features.to_csv(csv_path, index=False)
    print(f"Saved {csv_path}")
    return features


active_transforms = TRANSFORMS_MAIN + (TRANSFORMS_DERIVED_APPENDIX if RUN_DERIVED_APPENDIX_FEATURES else [])

if RUN_EEG_PILOT:
    pilot_subjects = select_balanced_pilot(participants, PILOT_SUBJECTS_PER_GROUP)
    epoch_features_pilot = extract_feature_table(
        pilot_subjects,
        regions=REGIONS,
        transforms=active_transforms,
        max_epochs_per_subject=MAX_EPOCHS_PER_SUBJECT_PILOT,
        output_name="epoch_features_pilot",
    )
else:
    epoch_features_pilot = pd.DataFrame()

if RUN_FULL_EEG_FEATURE_EXTRACTION:
    full_subjects = participants[participants["set_exists"]].reset_index(drop=True)
    epoch_features_full = extract_feature_table(
        full_subjects,
        regions=REGIONS,
        transforms=active_transforms,
        max_epochs_per_subject=MAX_EPOCHS_PER_SUBJECT_FULL,
        output_name="epoch_features_full",
    )
else:
    full_path = RESULTS_DIR / "epoch_features_full.csv"
    epoch_features_full = pd.read_csv(full_path) if full_path.exists() else pd.DataFrame()

epoch_features_pilot.head() if not epoch_features_pilot.empty else "Pilot extraction not run."

## 7. Aggregate Epoch Features To Subject-Level Matrix

Subject-level aggregation avoids treating multiple epochs from the same
participant as independent observations.


In [ ]:
def aggregate_subject_features(epoch_features: pd.DataFrame, output_name: str) -> pd.DataFrame:
    if epoch_features.empty:
        return pd.DataFrame()

    clean = epoch_features[
        epoch_features["feature_family"].isin(["psd", "hcfim"])
        & np.isfinite(epoch_features["value"])
    ].copy()

    def iqr(values):
        return np.nanpercentile(values, 75) - np.nanpercentile(values, 25)

    aggregated = (
        clean
        .groupby(["participant_id", "diagnosis", "feature_family", "feature_name"])["value"]
        .agg(mean="mean", std="std", median="median", iqr=iqr, p25=lambda x: np.nanpercentile(x, 25), p75=lambda x: np.nanpercentile(x, 75))
        .reset_index()
    )
    long_rows = []
    for row in aggregated.itertuples(index=False):
        for agg_name in ["mean", "std", "median", "iqr", "p25", "p75"]:
            long_rows.append({
                "participant_id": row.participant_id,
                "diagnosis": row.diagnosis,
                "feature_family": row.feature_family,
                "feature_name": f"{row.feature_name}__{agg_name}",
                "value": getattr(row, agg_name),
            })
    long_features = pd.DataFrame(long_rows)
    matrix = (
        long_features
        .pivot_table(index=["participant_id", "diagnosis"], columns="feature_name", values="value", aggfunc="first")
        .reset_index()
    )
    matrix.columns.name = None
    matrix.to_csv(RESULTS_DIR / f"{output_name}.csv", index=False)
    return matrix


subject_features_pilot = aggregate_subject_features(epoch_features_pilot, "subject_features_pilot")
subject_features_full = aggregate_subject_features(epoch_features_full, "subject_features_full")

subject_features_pilot.head() if not subject_features_pilot.empty else subject_features_full.head()

## 8. Geometric Separability Search

This section ranks HCFIM configurations by between-group separation and
within-group stability. The ranking happens at subject level.


In [ ]:
def parse_hcfim_feature_name(feature_name: str) -> dict:
    # hcfim__transform__distance__region__band__coord__agg
    parts = feature_name.split("__")
    if len(parts) < 7 or parts[0] != "hcfim":
        return {}
    return {
        "transform_id": parts[1],
        "distance": parts[2],
        "region": parts[3],
        "band": parts[4],
        "coord": parts[5],
        "agg": parts[6],
    }


def rank_hcfim_geometries(subject_features: pd.DataFrame, comparison_name: str, groups: list[str]) -> pd.DataFrame:
    if subject_features.empty:
        return pd.DataFrame()
    subset = subject_features[subject_features["diagnosis"].isin(groups)].copy()
    if subset["diagnosis"].nunique() != 2:
        return pd.DataFrame()

    hcfim_cols = [c for c in subset.columns if c.startswith("hcfim__") and c.endswith("__mean")]
    parsed = pd.DataFrame([{**parse_hcfim_feature_name(c), "column": c} for c in hcfim_cols])
    parsed = parsed.dropna(subset=["transform_id", "distance", "region", "band", "coord"], how="any")

    rows = []
    for keys, group_cols in parsed.groupby(["transform_id", "distance", "region", "band"]):
        coord_cols = {}
        for coord in ["H", "C", "FIM"]:
            match = group_cols[group_cols["coord"] == coord]
            if not match.empty:
                coord_cols[coord] = match.iloc[0]["column"]
        if set(coord_cols) != {"H", "C", "FIM"}:
            continue

        data = subset[["participant_id", "diagnosis", *coord_cols.values()]].dropna()
        if data["diagnosis"].nunique() != 2 or len(data) < 6:
            continue

        group_a, group_b = groups
        a = data[data["diagnosis"] == group_a][list(coord_cols.values())].to_numpy(float)
        b = data[data["diagnosis"] == group_b][list(coord_cols.values())].to_numpy(float)
        if len(a) < 2 or len(b) < 2:
            continue
        centroid_a = a.mean(axis=0)
        centroid_b = b.mean(axis=0)
        centroid_distance = float(np.linalg.norm(centroid_b - centroid_a))
        within_a = float(np.mean(np.linalg.norm(a - centroid_a, axis=1)))
        within_b = float(np.mean(np.linalg.norm(b - centroid_b, axis=1)))
        within_spread = np.nanmean([within_a, within_b])
        inter_intra_ratio = centroid_distance / within_spread if within_spread > 0 else np.nan

        labels = (data["diagnosis"] == group_b).astype(int).to_numpy()
        try:
            silhouette = silhouette_score(data[list(coord_cols.values())].to_numpy(float), labels)
        except Exception:
            silhouette = np.nan

        transform_id, distance, region, band = keys
        rows.append({
            "comparison": comparison_name,
            "control_group": group_a,
            "pathology_group": group_b,
            "transform_id": transform_id,
            "distance": distance,
            "region": region,
            "band": band,
            "n_subjects": len(data),
            "centroid_distance": centroid_distance,
            "within_spread": within_spread,
            "inter_intra_ratio": inter_intra_ratio,
            "silhouette": silhouette,
            "delta_H": float(centroid_b[0] - centroid_a[0]),
            "delta_C": float(centroid_b[1] - centroid_a[1]),
            "delta_FIM": float(centroid_b[2] - centroid_a[2]),
        })
    return pd.DataFrame(rows).sort_values(["inter_intra_ratio", "centroid_distance"], ascending=False)


subject_features_for_results = subject_features_full if not subject_features_full.empty else subject_features_pilot
all_rankings = []
for comparison_name, groups in CLASSIFICATION_COMPARISONS.items():
    ranking = rank_hcfim_geometries(subject_features_for_results, comparison_name, groups)
    if not ranking.empty:
        ranking.to_csv(RESULTS_DIR / f"hcfim_geometric_ranking_{comparison_name}.csv", index=False)
        all_rankings.append(ranking)

hcfim_rankings = pd.concat(all_rankings, ignore_index=True) if all_rankings else pd.DataFrame()
hcfim_rankings.head(20) if not hcfim_rankings.empty else "No geometric ranking available yet."


## 9. Plot Top-Ranked EEG HCFIM Spaces

After the full extraction is executed, this section plots the best HCFIM
configuration for each diagnostic comparison.


In [ ]:
def plot_top_hcfim_space(subject_features: pd.DataFrame, ranking_row: pd.Series, output_path: Path):
    transform_id = ranking_row["transform_id"]
    distance = ranking_row["distance"]
    region = ranking_row["region"]
    band = ranking_row["band"]
    groups = [ranking_row["control_group"], ranking_row["pathology_group"]]
    cols = {
        coord: f"hcfim__{transform_id}__{distance}__{region}__{band}__{coord}__mean"
        for coord in ["H", "C", "FIM"]
    }
    data = subject_features[subject_features["diagnosis"].isin(groups)][["participant_id", "diagnosis", *cols.values()]].dropna()
    data = data.rename(columns={v: k for k, v in cols.items()})
    if data.empty:
        return None

    colors = {"control": "#4C78A8", "alzheimer": "#ef3d3c", "frontotemporal_dementia": "#72B7B2"}
    colorsC = {"control": "#1e5a9b", "alzheimer": "#972726", "frontotemporal_dementia": "#158078"}
    fig = plt.figure(figsize=(9, 7))

    plt.rcParams.update({'font.size': 12})
    plt.rcParams.update({
        'axes.titlesize': 10,     # Title size
        'axes.labelsize': 11,     # X and Y label size
        'xtick.labelsize': 9,    # X-tick size
        'ytick.labelsize': 9,    # Y-tick size
        'legend.fontsize': 9,    # Legend size
        'figure.titlesize': 12    # Super title size
    })
    
    ax = fig.add_subplot(111, projection="3d")    
    #ax.set_facecolor("white")      # fondo del panel 3D
    #fig.patch.set_facecolor("white") # fondo general de la figura
    
    for diagnosis, group in data.groupby("diagnosis"):
        ax.scatter(group["H"], group["C"], group["FIM"], s=55, alpha=0.70,
                   c=colors.get(diagnosis, "#666666"), label=diagnosis)
        centroid = group[["H", "C", "FIM"]].mean()
        ax.scatter(centroid["H"], centroid["C"], centroid["FIM"],
                   marker="X", s=190, c=colorsC.get(diagnosis, "#666666"),
                   edgecolors="white", linewidths=1.2, label=f"centroid {diagnosis}")
        
    ax.set_xlabel("H")
    ax.set_ylabel("C")
    ax.set_zlabel("FIM")

    ax.xaxis.pane.set_facecolor((1, 1, 1, 1))
    ax.yaxis.pane.set_facecolor((1, 1, 1, 1))
    ax.zaxis.pane.set_facecolor((1, 1, 1, 1))

    ax.xaxis.set_major_formatter(FormatStrFormatter("%.4f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.4f"))
    ax.zaxis.set_major_formatter(FormatStrFormatter("%.4f"))
    
    ax.view_init(elev=24, azim=-58)
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
    
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return output_path

if not hcfim_rankings.empty:
    for comparison_name, ranking in hcfim_rankings.groupby("comparison"):
        top = ranking.iloc[0]
        path = FIGURES_DIR / f"hcfim_space_top_{comparison_name}.png"
        plot_top_hcfim_space(subject_features_for_results, top, path)
else:
    print("Run feature extraction and geometric ranking before plotting EEG HCFIM spaces.")

## 10. Classification And Ablations

Classification is downstream evidence of utility. The main methodological
claim remains the information-theoretic geometry. Feature selection, scaling
and imputation are always inside the validation pipeline.


In [ ]:
def feature_columns_for_family(df: pd.DataFrame, feature_family: str) -> list[str]:
    if feature_family == "psd":
        return [c for c in df.columns if c.startswith("psd__")]
    if feature_family == "hcfim":
        return [c for c in df.columns if c.startswith("hcfim__")]
    if feature_family == "psd_plus_hcfim":
        return [c for c in df.columns if c.startswith("psd__") or c.startswith("hcfim__")]
    raise ValueError(f"Unknown feature family: {feature_family}")


def classifier_grid():
    return {
        "logistic_elasticnet": (
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
            {
                "model__C": [0.05, 0.1, 0.5, 1.0],
                "model__l1_ratio": [0.15, 0.5, 0.85],
            },
        ),
        "svm_rbf": (
            SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
            {
                "model__C": [0.1, 1.0, 10.0],
                "model__gamma": ["scale", 0.01, 0.1],
            },
        ),
        "random_forest": (
            RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
            {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3, None],
            },
        ),
        "extra_trees": (
            ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE),
            {
                "model__n_estimators": [300],
                "model__max_depth": [2, 3, None],
            },
        ),
        "gradient_boosting": (
            GradientBoostingClassifier(random_state=RANDOM_STATE),
            {
                "model__n_estimators": [50, 100],
                "model__learning_rate": [0.03, 0.1],
                "model__max_depth": [1, 2],
            },
        ),
        "mlp_small": (
            MLPClassifier(
                hidden_layer_sizes=(8,),
                alpha=0.01,
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
            {
                "model__alpha": [0.001, 0.01, 0.1],
                "model__hidden_layer_sizes": [(8,), (12,)],
            },
        ),
    }


def evaluate_feature_family(df: pd.DataFrame, comparison_name: str, groups: list[str], feature_family: str) -> pd.DataFrame:
    subset = df[df["diagnosis"].isin(groups)].copy()
    feature_cols = feature_columns_for_family(subset, feature_family)
    if len(feature_cols) == 0 or subset["diagnosis"].nunique() != 2:
        return pd.DataFrame()

    X = subset[feature_cols].to_numpy(float)
    y = (subset["diagnosis"] == groups[1]).astype(int).to_numpy()

    outer_cv = StratifiedKFold(n_splits=min(5, np.bincount(y).min()), shuffle=True, random_state=RANDOM_STATE)
    model_defs = classifier_grid()
    rows = []

    for model_name, (model, grid) in model_defs.items():
        oof_prob = np.full(len(y), np.nan)
        oof_pred = np.full(len(y), np.nan)
        fold_rows = []
        for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
            k = min(30, max(3, X.shape[1] // 10), X.shape[1])
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("select", SelectKBest(score_func=f_classif, k=k)),
                ("model", model),
            ])
            inner_cv = StratifiedKFold(n_splits=min(3, np.bincount(y[train_idx]).min()), shuffle=True, random_state=RANDOM_STATE + fold)
            search = GridSearchCV(
                pipe,
                param_grid=grid,
                scoring="roc_auc",
                cv=inner_cv,
                n_jobs=-1,
                error_score=np.nan,
            )
            search.fit(X[train_idx], y[train_idx])
            best = search.best_estimator_
            if hasattr(best, "predict_proba"):
                prob = best.predict_proba(X[test_idx])[:, 1]
            else:
                scores = best.decision_function(X[test_idx])
                prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
            pred = (prob >= 0.5).astype(int)
            oof_prob[test_idx] = prob
            oof_pred[test_idx] = pred
            fold_rows.append({
                "fold": fold,
                "best_params": search.best_params_,
                "best_inner_auc": search.best_score_,
            })

        valid = np.isfinite(oof_prob)
        tn, fp, fn, tp = confusion_matrix(y[valid], oof_pred[valid], labels=[0, 1]).ravel()
        rows.append({
            "comparison": comparison_name,
            "feature_family": feature_family,
            "model": model_name,
            "n_subjects": int(valid.sum()),
            "n_features": len(feature_cols),
            "roc_auc": roc_auc_score(y[valid], oof_prob[valid]),
            "balanced_accuracy": balanced_accuracy_score(y[valid], oof_pred[valid]),
            "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "f1": f1_score(y[valid], oof_pred[valid]),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        })

        pred_df = pd.DataFrame({
            "participant_id": subset["participant_id"].to_numpy()[valid],
            "diagnosis": subset["diagnosis"].to_numpy()[valid],
            "y_true": y[valid],
            "y_prob": oof_prob[valid],
            "y_pred": oof_pred[valid].astype(int),
            "comparison": comparison_name,
            "feature_family": feature_family,
            "model": model_name,
        })
        pred_df.to_csv(RESULTS_DIR / f"oof_predictions_{comparison_name}_{feature_family}_{model_name}.csv", index=False)
    return pd.DataFrame(rows)


classification_results = []
if RUN_FULL_CLASSIFICATION and not subject_features_full.empty:
    for comparison_name, groups in CLASSIFICATION_COMPARISONS.items():
        for family in ["psd", "hcfim", "psd_plus_hcfim"]:
            result = evaluate_feature_family(subject_features_full, comparison_name, groups, family)
            if not result.empty:
                classification_results.append(result)
    classification_results = pd.concat(classification_results, ignore_index=True)
    classification_results.to_csv(RESULTS_DIR / "classification_results.csv", index=False)
else:
    results_path = RESULTS_DIR / "classification_results.csv"
    classification_results = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame()

classification_results.sort_values("roc_auc", ascending=False).head(20) if not classification_results.empty else "Classification not run yet."


## 11. Classification Figures

This section converts stored out-of-fold predictions into ROC curves and
confusion matrices for the Results section.


In [ ]:
classification_results.sort_values(by="roc_auc", ascending=False)

In [ ]:
def plot_roc_and_confusion_for_best(results: pd.DataFrame):
    if results.empty:
        print("No classification results available.")
        return []

    saved = []
    best_rows = (
        results
        .sort_values(["roc_auc"], ascending=False)
        .groupby("comparison", as_index=False)
        .head(1)
    )

    print(best_rows)
    
    for row in best_rows.itertuples(index=False):
        pred_path = RESULTS_DIR / f"oof_predictions_{row.comparison}_{row.feature_family}_{row.model}.csv"
        if not pred_path.exists():
            continue
        pred = pd.read_csv(pred_path)
        fpr, tpr, _ = roc_curve(pred["y_true"], pred["y_prob"])
        cm = confusion_matrix(pred["y_true"], pred["y_pred"], labels=[0, 1]).astype(int)
        row_tot = cm.sum(axis=1, keepdims=True)
        cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100.0

        annot = np.array([[f"{int(cm[i, j])}/{int(row_tot[i, 0])}\n({cm_pct[i, j]:.1f}%)"
                   for j in range(2)] for i in range(2)])

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
        axes[0].plot(fpr, tpr, color="#2F5597", linewidth=2.2, label=f"AUC = {row.roc_auc:.3f}")
        axes[0].plot([0, 1], [0, 1], linestyle="--", color="#999999", linewidth=1.0)
        axes[0].set_xlabel("False positive rate")
        axes[0].set_ylabel("True positive rate")
        axes[0].legend(loc="lower right")

        sns.heatmap(
            cm_pct,
            annot=annot,
            fmt="",
            cmap="Blues",
            cbar=False,
            xticklabels=["control", "pathology"],
            yticklabels=["control", "pathology"],
            ax=axes[1],
        )
        axes[1].set_xlabel("Predicted")
        axes[1].set_ylabel("Observed")
        axes[1].set_title(f"Threshold = 0.50 | n = {int(cm.sum())}")

        output_path = FIGURES_DIR / f"classification_best_{row.comparison}.png"
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.show()
        saved.append(output_path)
    return saved

In [ ]:
classification_figure_paths = plot_roc_and_confusion_for_best(classification_results)
classification_figure_paths

## 12. Takeaways Template

Complete this section only after executing the full analysis.

Suggested result narrative:

1. Synthetic regimes occupied distinguishable regions of the induced
   probability simplex and of HC/HFIM/HCFIM spaces.
2. EEG group separation depended on probability transformation, band and
   cortical region.
3. HCFIM-only features should be compared against PSD-only and PSD+HCFIM
   ablations rather than claimed as universally superior.
4. Classification metrics must be reported with subject-level validation and
   confidence intervals.

Saved outputs:

```text
figures/figure_1_methods_synthetic_geometry.png
results/participants_index.csv
results/participants_summary.csv
results/epoch_features_pilot.csv
results/epoch_features_full.csv
results/subject_features_full.csv
results/hcfim_geometric_ranking_*.csv
results/classification_results.csv
```
